# STEP 3 — Hybrid IR System: TF-IDF + BM25 + Calorie-Aware Ranking

Membangun sistem temu kembali dengan:
1. **TF-IDF** — relevansi kata kunci
2. **BM25** — ranking dokumen probabilistik
3. **Calorie-Aware Score** — bobot kalori rendah
4. **Hybrid Score** = α·TF-IDF + β·BM25 + γ·Calorie

**Input:** `data/dataset_final_resep_nutrisi.csv`

**Output:** `src/ir_model.pkl`

In [1]:
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm import tqdm

tqdm.pandas()
print("Library siap.")

Library siap.


In [2]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("../data/dataset_final_resep_nutrisi.csv")

print(f"Total resep: {len(df):,}")
print("Kolom:", df.columns.tolist())
df.head(3)

Total resep: 9,873
Kolom: ['Title', 'Ingredients', 'Steps', 'Loves', 'Kategori', 'calories', 'proteins', 'fat', 'carbohydrate', 'calories_total', 'porsi', 'matched_ingredients', 'total_ingredients', 'match_ratio', 'matched_names']


,Title,Ingredients,Steps,Loves,Kategori,calories,proteins,fat,carbohydrate,calories_total,porsi,matched_ingredients,total_ingredients,match_ratio,matched_names
0,Nasi Briyani Simple Ala Mamah Mumtaz,500 g daging sapi atau ayam atau kambing juga ...,Siapkan semua bahan-bahannya.--Masak air sekit...,105,kambing.csv,405.7,24.88,5.09,66.53,1622.7,4.0,9,29,0.310,"bawang bombay, dadu(80.0g), cm jahe(200.0g), b..."
1,Bakso Sapi Simple Ga Ribet,500 gram daging sapi--85 gram es batu / 17%--1...,Daging sapi dipotong kecil-kecil supaya mudah ...,21,sapi.csv,483.6,43.50,29.25,15.79,1934.3,4.0,7,12,0.583,"daging sapi(500.0g), tepung tapioka(100.0g), p..."
2,Kroket kentang isi ayam dan wortel\n(Indonesia...,Bahan kulit:--1 buah kuning telur--1/4 buah pa...,"Membuat kulit kroket\nSiapkan wajan, beri miny...",3,ayam.csv,477.8,26.59,14.08,62.25,1911.3,4.0,11,30,0.367,"kuning telur(80.0g), sendok makan susu kental ..."


In [3]:
# ============================================================
# SETUP STEMMER & STOPWORD (PySastrawi)
# ============================================================

factory_stemmer  = StemmerFactory()
factory_stopword = StopWordRemoverFactory()

stemmer          = factory_stemmer.create_stemmer()
stopword_remover = factory_stopword.create_stop_word_remover()

STOPWORDS_TAMBAHAN = {
    "resep", "cara", "membuat", "masak", "memasak", "buat",
    "mudah", "enak", "lezat", "sedap", "nikmat",
    "praktis", "simpel", "simple", "cepat",
    "yummy", "mantap", "yuk", "ayo",
}


def preprocess_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = stopword_remover.remove(text)
    text = stemmer.stem(text)
    words = [w for w in text.split() if w not in STOPWORDS_TAMBAHAN]
    return " ".join(words)


# Test
print("Test preprocessing:")
for q in ["resep ayam yang rendah kalori", "masakan ikan tidak bikin gemuk"]:
    print(f"  '{q}'")
    print(f"  → '{preprocess_text(q)}'")

Test preprocessing:
  'resep ayam yang rendah kalori'
  → 'ayam rendah kalori'
  'masakan ikan tidak bikin gemuk'
  → 'masakan ikan bikin gemuk'


In [4]:
# ============================================================
# BUILD DOKUMEN TEKS
# Title diulang 3x agar lebih dominan saat matching
# ============================================================

print("Membangun dan preprocessing dokumen...")
print("(Estimasi waktu: 2-5 menit)\n")

df["doc_text_raw"] = (
    df["Title"].fillna("") + " " +
    df["Title"].fillna("") + " " +
    df["Title"].fillna("") + " " +
    df["Ingredients"].fillna("")
)

df["doc_text_processed"] = df["doc_text_raw"].progress_apply(preprocess_text)

print("\nSelesai.")
df[["Title", "doc_text_processed"]].head(3)

Membangun dan preprocessing dokumen...
(Estimasi waktu: 2-5 menit)



100%|████████████████████████████████████████████████████████████████████████████████| 9873/9873 [00:10<00:00, 921.38it/s]


Selesai.


,Title,doc_text_processed
0,Nasi Briyani Simple Ala Mamah Mumtaz,nasi briyani ala mamah mumtaz nasi briyani ala...
1,Bakso Sapi Simple Ga Ribet,bakso sapi ga ribet bakso sapi ga ribet bakso ...
2,Kroket kentang isi ayam dan wortel\n(Indonesia...,kroket kentang isi ayam wortel indonesian pota...


In [5]:
# ============================================================
# BUILD TF-IDF INDEX
# ============================================================

print("Building TF-IDF index...")

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

corpus       = df["doc_text_processed"].tolist()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print(f"TF-IDF matrix : {tfidf_matrix.shape}")
print(f"Vocabulary    : {len(tfidf_vectorizer.vocabulary_):,} terms")

Building TF-IDF index...
TF-IDF matrix : (9873, 24267)
Vocabulary    : 24,267 terms


In [6]:
# ============================================================
# BUILD BM25 INDEX
# ============================================================

print("Building BM25 index...")

tokenized_corpus = [doc.split() for doc in corpus]
bm25_index = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print(f"BM25 siap. Corpus: {len(tokenized_corpus):,} dokumen")

Building BM25 index...
BM25 siap. Corpus: 9,873 dokumen


In [7]:
# ============================================================
# CALORIE-AWARE SCORE
# ============================================================

KALORI_THRESHOLD = 500.0


def hitung_calorie_score(calories) -> float:
    if pd.isna(calories) or calories <= 0:
        return 0.5
    return round(1.0 - min(calories / KALORI_THRESHOLD, 1.0), 4)


df["calorie_score"] = df["calories"].apply(hitung_calorie_score)

print("Calorie score sample:")
print(df[["Title", "calories", "calorie_score"]].head(10).to_string())

Calorie score sample:
                                                                                        Title  calories  calorie_score
0                                                        Nasi Briyani Simple Ala Mamah Mumtaz     405.7         0.1886
1                                                                  Bakso Sapi Simple Ga Ribet     483.6         0.0328
2  Kroket kentang isi ayam dan wortel\n(Indonesian Potato Croquettes with Chicken and carrot)     477.8         0.0444
3                            Sate Telur Gulung Crispy Ekonomis Isi Sosis | 1 Resep Jadi 24pcs     125.2         0.7496
4                                                                        Siomay Ayam ekonomis     622.0         0.0000
5                                                                        Martabak Gule Daging     660.0         0.0000
6                                                                Nasi Kebuli Kambing Nikmat !     413.8         0.1724
7                         

In [8]:
# ============================================================
# INTENT DETECTION
# ============================================================

KATA_DIET = [
    "diet", "rendah kalori", "rendah lemak", "sehat", "ringan",
    "tidak gemuk", "nggak gemuk", "gak gemuk", "ga bikin gemuk",
    "tidak bikin gemuk", "langsing", "kurus", "turun berat",
    "berat badan", "low calorie", "low fat", "light",
    "kalori rendah", "lemak rendah",
]


def deteksi_intent_diet(query: str) -> bool:
    return any(kata in query.lower() for kata in KATA_DIET)


print("Test intent:")
for q in ["resep ayam rendah kalori", "masakan ikan yang enak", "makanan nggak gemuk"]:
    print(f"  '{q}' → {deteksi_intent_diet(q)}")

Test intent:
  'resep ayam rendah kalori' → True
  'masakan ikan yang enak' → False
  'makanan nggak gemuk' → True


In [9]:
# ============================================================
# FUNGSI HYBRID SEARCH
# ============================================================

def hybrid_search(
    query: str,
    top_k: int = 10,
    alpha: float = 0.4,
    beta: float = 0.4,
    gamma: float = 0.2,
    kalori_max: float = None,
) -> pd.DataFrame:

    query_processed = preprocess_text(query)
    if not query_processed.strip():
        return pd.DataFrame()

    # TF-IDF
    query_vec    = tfidf_vectorizer.transform([query_processed])
    tfidf_scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    tfidf_max    = tfidf_scores.max()
    tfidf_norm   = tfidf_scores / tfidf_max if tfidf_max > 0 else tfidf_scores

    # BM25
    bm25_scores = bm25_index.get_scores(query_processed.split())
    bm25_max    = bm25_scores.max()
    bm25_norm   = bm25_scores / bm25_max if bm25_max > 0 else bm25_scores

    # Calorie
    cal_scores = df["calorie_score"].values

    # Adaptive weights
    if deteksi_intent_diet(query):
        a, b, g = 0.35, 0.35, 0.30
    else:
        a, b, g = alpha, beta, gamma

    # Hybrid
    hybrid = a * tfidf_norm + b * bm25_norm + g * cal_scores

    # Filter kalori
    if kalori_max is not None:
        mask = (df["calories"].notna() & (df["calories"] <= kalori_max)).values
        hybrid = hybrid * mask

    # Top-K
    top_idx = np.argsort(hybrid)[::-1][:top_k]
    hasil   = df.iloc[top_idx].copy()
    hasil["tfidf_score"]  = tfidf_norm[top_idx]
    hasil["bm25_score"]   = bm25_norm[top_idx]
    hasil["hybrid_score"] = hybrid[top_idx]

    return hasil[hasil["hybrid_score"] > 0].reset_index(drop=True)


print("Fungsi hybrid_search siap.")

Fungsi hybrid_search siap.


In [10]:
# ============================================================
# TEST HYBRID SEARCH
# ============================================================

print("TEST 1: resep ayam rendah kalori")
h1 = hybrid_search("resep ayam rendah kalori", top_k=5)
print(h1[["Title", "Kategori", "calories", "hybrid_score"]].to_string())

print("\nTEST 2: saya punya tahu dan tempe")
h2 = hybrid_search("saya punya tahu dan tempe", top_k=5)
print(h2[["Title", "Kategori", "calories", "hybrid_score"]].to_string())

print("\nTEST 3: ikan sehat kalori max 300")
h3 = hybrid_search("ikan sehat", top_k=5, kalori_max=300)
print(h3[["Title", "Kategori", "calories", "hybrid_score"]].to_string())

TEST 1: resep ayam rendah kalori
                                                           Title   Kategori  calories  hybrid_score
0    Sambel Tempe Kukus Diet Rendah Kalori Rice Cooker Anak kost  tempe.csv      54.1      0.967248
1  Tempe Bacem Diet Rendah Kalori Rice Cooker Anak Kost (No Oil)  tempe.csv     126.4      0.908628
2                          Ayam Geprek Diet Fatsecret kalori 206   ayam.csv     145.3      0.686753
3                       Tahu Kukus rendah lemak cocok untuk diet   tahu.csv     164.7      0.507992
4                                                  Tahu Keriting   tahu.csv     269.0      0.395701

TEST 2: saya punya tahu dan tempe
                  Title   Kategori  calories  hybrid_score
0  Tempe goreng ori...🍡  tempe.csv     157.0      0.918936
1     Tahu tempe goreng  tempe.csv      32.4      0.897355
2  Krenseng tahu tempe.   tahu.csv      97.0      0.895628
3   Tempe goreng simply  tempe.csv      78.2      0.860742
4          Lumpia Tempe  tempe.csv   

In [11]:
# ============================================================
# SIMPAN MODEL
# ============================================================

model_bundle = {
    "tfidf_vectorizer":  tfidf_vectorizer,
    "tfidf_matrix":      tfidf_matrix,
    "bm25_index":        bm25_index,
    "df":                df,
    "calorie_threshold": KALORI_THRESHOLD,
}

import os
os.makedirs("../src", exist_ok=True)

with open("../src/ir_model.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Model saved: src/ir_model.pkl")

Model saved: src/ir_model.pkl
